# Build `ouroboros-cache` Kaggle Dataset

Run this notebook once on Kaggle with internet enabled. It will:

1. Download the full Hugging Face cache for `ai21labs/AI21-Jamba-Reasoning-3B`
2. Build the T4 (`sm75`) wheels for `causal_conv1d` and `mamba_ssm`
3. Assemble `ouroboros_cache/` so it can be published as a Kaggle dataset input

Expected dataset layout after the final cell:

```
/kaggle/working/ouroboros_cache/
  hf_cache/
  wheels/
```

Before running:
- Add a Kaggle secret named `HF_TOKEN`
- Enable internet for the notebook
- Use a T4 session so the built wheels match `sm75`


In [ ]:
import os
from pathlib import Path
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
os.environ['HF_TOKEN'] = user_secrets.get_secret("HF_TOKEN")

os.environ['HF_HOME'] = '/kaggle/working/hf_cache'
Path(os.environ['HF_HOME']).mkdir(parents=True, exist_ok=True)

hf_token = os.environ.get('HF_TOKEN')
if not hf_token:
    raise RuntimeError('Missing HF_TOKEN. Add it as a Kaggle secret before running this notebook.')

## 1. Download the model into the Hugging Face cache

This uses `snapshot_download` to populate the cache in a reusable HF layout.


In [ ]:
from huggingface_hub import snapshot_download

target_dir = '/kaggle/working/hf_cache/hub/models--ai21labs--AI21-Jamba-Reasoning-3B'
snapshot_download(
    repo_id='ai21labs/AI21-Jamba-Reasoning-3B',
    token=os.environ['HF_TOKEN'],
    local_dir=target_dir,
    local_dir_use_symlinks=False,
)
print('Model download complete.')
print('Hub contents:', os.listdir('/kaggle/working/hf_cache/hub'))


## Optional verification load

Uncomment and run this cell only if you want to verify the cache by doing a real model load. It is not required for dataset assembly.


In [ ]:
# import torch
# from transformers import AutoModelForCausalLM, BitsAndBytesConfig
#
# model = AutoModelForCausalLM.from_pretrained(
#     'ai21labs/AI21-Jamba-Reasoning-3B',
#     quantization_config=BitsAndBytesConfig(
#         load_in_4bit=True,
#         bnb_4bit_quant_type='nf4',
#         bnb_4bit_compute_dtype=torch.float16,
#         bnb_4bit_use_double_quant=True,
#     ),
#     device_map={'': 0},
#     trust_remote_code=True,
#     low_cpu_mem_usage=True,
# )
# print('Cache populated:', os.listdir('/kaggle/working/hf_cache/hub'))
# del model
# torch.cuda.empty_cache()


## 2. Build the T4 wheels (`sm75`)

This matches the bootstrap logic used by the training notebook, but collects the wheels into one persistent dataset.


In [ ]:
import subprocess, sys, shutil, os
from pathlib import Path
from huggingface_hub import hf_hub_download, HfApi

HF_TOKEN = os.environ["HF_TOKEN"]
HUB_REPO  = "WeirdRunner/Ouroboros"
WHEEL_OUT = Path("/kaggle/working/wheels")
WHEEL_OUT.mkdir(exist_ok=True)

import torch
cc = torch.cuda.get_device_capability()
ARCH_SUFFIX = f"sm{cc[0]}{cc[1]}"   # e.g. "sm75"
ARCH_LIST   = f"{cc[0]}.{cc[1]}+PTX"
print(f"GPU arch: {ARCH_SUFFIX}")

WHEEL_SPECS = [
    ("causal_conv1d", "causal-conv1d==1.6.1",
     "causal_conv1d-1.6.1-cp312-cp312-linux_x86_64"),
    ("mamba_ssm",     "git+https://github.com/state-spaces/mamba.git@v1.2.2",
     "mamba_ssm-1.2.2-cp312-cp312-linux_x86_64"),
]

for pkg_name, pip_spec, base in WHEEL_SPECS:
    hub_filename  = f"{base}-{ARCH_SUFFIX}.whl"
    local_path    = WHEEL_OUT / f"{pkg_name}.whl"
    downloaded    = False

    # ── Try Hub first ──────────────────────────────────────────────────────
    try:
        dl = hf_hub_download(
            repo_id=HUB_REPO, filename=hub_filename,
            token=HF_TOKEN, local_dir=str(WHEEL_OUT),
        )
        shutil.copy2(dl, str(local_path))
        print(f"✓ Downloaded {hub_filename} from Hub — skipping compilation")
        downloaded = True
    except Exception as e:
        print(f"  Hub miss ({hub_filename}): {e}")

    # ── Compile from source only if Hub miss ───────────────────────────────
    if not downloaded:
        print(f"  Compiling {pip_spec} for {ARCH_SUFFIX} ...")
        env = {**os.environ, "TORCH_CUDA_ARCH_LIST": ARCH_LIST, "MAX_JOBS": "4"}
        subprocess.run(
            [sys.executable, "-m", "pip", "wheel", pip_spec,
             "--no-build-isolation", "--no-deps", "-w", str(WHEEL_OUT)],
            env=env, check=True,
        )
        built = max(WHEEL_OUT.glob(f"{pkg_name}*.whl"), key=lambda p: p.stat().st_mtime)
        shutil.copy2(str(built), str(local_path))
        print(f"  Built: {built.name}")

        # Upload arch-specific wheel to Hub so future sessions skip compilation
        arch_path = WHEEL_OUT / hub_filename
        shutil.copy2(str(local_path), str(arch_path))
        try:
            HfApi(token=HF_TOKEN).upload_file(
                path_or_fileobj=str(arch_path),
                path_in_repo=hub_filename,
                repo_id=HUB_REPO, token=HF_TOKEN,
                commit_message=f"Add wheel: {hub_filename}",
            )
            print(f"  Uploaded {hub_filename} to Hub ✓")
        except Exception as ue:
            print(f"  [warn] Hub upload failed: {ue}")

print("\nFinal wheels:")
for w in sorted(WHEEL_OUT.glob("*.whl")):
    print(f"  {w.name}")

## 3. Assemble the Kaggle dataset directory


import shutil
from pathlib import Path

out = Path('/kaggle/working/ouroboros_cache')
(out / 'wheels').mkdir(parents=True, exist_ok=True)

for whl in Path('/kaggle/working/wheels').glob('*.whl'):
    shutil.copy2(whl, out / 'wheels' / whl.name)

shutil.copytree(
    '/kaggle/working/hf_cache',
    str(out / 'hf_cache'),
    dirs_exist_ok=True,
)

print('Cache assembled at:', out)
for path in sorted(out.rglob('*')):
    print(path)


## 4. Publish it as a Kaggle dataset

In the notebook Output tab:
1. Click **New dataset**
2. Name it `ouroboros-cache`
3. Make it **Private** or **Unlisted** as needed
4. Publish the `ouroboros_cache/` directory

Expected slug for Account A:
- `weirdrunner/ouroboros-cache`

Then attach it to `kaggle-utils.ipynb` as an input so it mounts at `/kaggle/input/ouroboros-cache/`.
